In [11]:
%pip install --upgrade "protobuf<=5.29.4" --quiet
%pip install pinecone

Note: you may need to restart the kernel to use updated packages.


DEPRECATION: Loading egg at c:\users\cheth\appdata\local\programs\python\python311\lib\site-packages\ntlflowlyzer-0.1.0-py3.11.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


DEPRECATION: Loading egg at c:\users\cheth\appdata\local\programs\python\python311\lib\site-packages\ntlflowlyzer-0.1.0-py3.11.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
%pip install firebase-admin

   ---------------------------------------- 0.0/3.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.4 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.4 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.4 MB ? eta -:--:--
   ------ --------------------------------- 0.5/3.4 MB 578.7 kB/s eta 0:00:05
   ------ --------------------------------- 0.5/3.4 MB 578.7 kB/s eta 0:00:05
   --------- ------------------------------ 0.8/3.4 MB 657.8 kB/s eta 0:00:04
   --------- ------------------------------ 0.8/3.4 MB 657.8 kB/s eta 0:00:04
   ------------ --------------------------- 1.0/3.4 MB 680.3 kB/s eta 0:00:04
   --------------- ------------------------ 1.3/3.4 MB 714.3 kB/s eta 0:00:03
   ------------------ --------------------- 1.6/3.4 MB 783.9 kB/s eta 0:00:03
   ------------------ --------------------- 1.6/3.4 MB 783.9 kB/s eta 0:00:03
   ------------------ ---------

DEPRECATION: Loading egg at c:\users\cheth\appdata\local\programs\python\python311\lib\site-packages\ntlflowlyzer-0.1.0-py3.11.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Pinecone credentials
PINECONE_API_KEY = "pcsk_2it9oG_RzRNfQdLGg9jUen7wW6viE9JpLRgVTHtWbTZhomuZKmhuyYnrh8GgMyrHJMz37Q"
PINECONE_CASE_DENSE_INDEX = f"law-cases-dense"
PINECONE_CASE_SPARSE_INDEX = f"law-cases-sparse"
PINECONE_ACTS_DENSE_INDEX = f"law-acts-dense"
PINECONE_ACTS_SPARSE_INDEX = f"law-acts-sparse"
PINECONE_DENSE_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
PINECONE_SPARSE_EMBED_MODEL = "naver/splade-cocondenser-ensembledistil"

# Neo4j credentials and URI
NEO4J_URI = "neo4j+s://66d16355.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "G4UXZ6KLGd1dzo57rp6ITypJHZ37aM1fn-exAWdw3p8"
NEO4J_DATABASE = "neo4j"

In [ ]:
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

# Initialize Pinecone with the credentials
pc = Pinecone(
    api_key=PINECONE_API_KEY,
)

# Connect to the index
dense_cases_index = pc.Index(PINECONE_CASE_DENSE_INDEX)
sparse_cases_index = pc.Index(PINECONE_CASE_SPARSE_INDEX)
dense_acts_index = pc.Index(PINECONE_ACTS_DENSE_INDEX)
sparse_acts_index = pc.Index(PINECONE_ACTS_SPARSE_INDEX)

dense_model = SentenceTransformer(PINECONE_DENSE_EMBED_MODEL)

tokenizer = AutoTokenizer.from_pretrained(PINECONE_SPARSE_EMBED_MODEL)
sparse_model = AutoModelForMaskedLM.from_pretrained(PINECONE_SPARSE_EMBED_MODEL)
sparse_model.eval()

# Test the connection by getting index statistics
index_cases_dense = dense_cases_index.describe_index_stats()
print(f"Index stats: {index_cases_dense}\n")
index_cases_sparse = sparse_cases_index.describe_index_stats()
print(f"Index stats: {index_cases_sparse}\n")
index_acts_dense = dense_acts_index.describe_index_stats()
print(f"Index stats: {index_acts_dense}\n")
index_acts_sparse = sparse_acts_index.describe_index_stats()
print(f"Index stats: {index_acts_sparse}\n")


def embed_text_dense(text):
    """
    Embed the given text using a pre-trained model
    
    Args:
        text (str): The text to embed
        
    Returns:
        list: List of embeddings
    """
    try:
        # Use a pre-trained model to embed the text
        embeddings = dense_model.encode(text, convert_to_tensor=True)
        return embeddings.tolist()
    except Exception as e:
        print(f"Error embedding text: {e}")
        return []
    
def embed_text_sparse(text):
    """
    Embed the given text using a pre-trained model for sparse embeddings
    
    Args:
        text (str): The text to embed
        
    Returns:
        list: List of sparse embeddings
    """
    try:
        # Use a pre-trained model to embed the text
        inputs = tokenizer(text, return_tensors="pt")
        with torch.no_grad():
            outputs = sparse_model(**inputs).logits.squeeze(0)
        scores = torch.log(1 + torch.relu(outputs))
        max_scores, _ = torch.max(scores, dim=0)
        non_zero_indices = torch.nonzero(max_scores).squeeze(1).tolist()
        non_zero_values = max_scores[non_zero_indices].tolist()
        tokens = tokenizer.convert_ids_to_tokens(non_zero_indices)
        indices = tokenizer.convert_tokens_to_ids(tokens)
        return {
            "values": non_zero_values,
            "indices": indices,
        }
    except Exception as e:
        print(f"Error embedding text: {e}")
        return []

def query_vectorDB(query_text, type, doc, top_k=5):
    """
    Query Pinecone index with the given text
    
    Args:
        query_text (str): The text to search for
        top_k (int): Number of results to return
        
    Returns:
        list: List of matching vectors with their scores
    """
    try:
        if type == 'dense':
            query_vector = embed_text_dense(query_text)

            index = dense_cases_index if doc == 'case' else dense_acts_index
            query_response = index.query(
                vector=query_vector,
                top_k=top_k,
                include_metadata=True,
            )
        elif type == 'sparse':
            query_vector = embed_text_sparse(query_text)

            index = sparse_cases_index if doc == 'case' else sparse_acts_index
            query_response = index.query(
                sparse_vector=query_vector,
                top_k=top_k,
                include_metadata=True
            )
        else:
            raise ValueError("Invalid type specified. Use 'dense' or 'sparse'.")
        results = query_response['matches']

        if not results: 
            print("No results found.")
            return []
        print(f"Found {len(results)} results for query: '{query_text}'\n")
        # Return the matches from the query response
        return results
    except Exception as e:
        print(f"Error querying Pinecone: {e}")
        return []

In [54]:
import asyncio
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Union, Any
from enum import Enum
import numpy as np
from abc import ABC, abstractmethod
import networkx as nx
from sentence_transformers import SentenceTransformer
import logging
from datetime import datetime, timedelta
import json
import re
from transformers import AutoTokenizer, AutoModelForMaskedLM
from neo4j import GraphDatabase
from pinecone import Pinecone
import torch
import nest_asyncio
import firebase_admin

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class QueryIntent(Enum):
    DOCTRINAL = "doctrinal"
    PRECEDENTIAL = "precedential"
    PROCEDURAL = "procedural"
    COMPARATIVE = "comparative"
    MIXED = "mixed"

class DocumentType(Enum):
    CASE = "case"
    ACT = "act"

@dataclass
class QueryContext:
    raw_query: str
    intent: QueryIntent
    complexity_score: float
    legal_domains: List[str]
    jurisdictions: List[str]
    temporal_constraints: Optional[Dict[str, Any]] = None
    extracted_entities: List[str] = field(default_factory=list)

@dataclass
class DocumentMetadata:
    doc_id: str
    doc_type: DocumentType
    title: str
    court_level: Optional[int] = None  # 1=Supreme, 2=High, 3=District
    jurisdiction: str = ""
    date: Optional[datetime] = None
    legal_domains: List[str] = field(default_factory=list)
    act_sections: List[str] = field(default_factory=list)

@dataclass
class RetrievedChunk:
    doc_id: str
    chunk_id: str
    content: str
    metadata: DocumentMetadata
    vector_score: float
    chunk_index: int
    
class KGFeatures:
    def __init__(self):
        self.pagerank_score: float = 0.0
        self.betweenness_centrality: float = 0.0
        self.citation_count: int = 0
        self.recent_citation_boost: float = 0.0
        self.jurisdictional_weight: float = 0.0
        self.cross_domain_citations: int = 0
        self.authority_score: float = 0.0

@dataclass
class EnhancedChunk(RetrievedChunk):
    kg_features: KGFeatures = field(default_factory=KGFeatures)
    final_score: float = 0.0

@dataclass
class ReasoningStep:
    step_type: str  # "major_premise", "minor_premise", "application", "conclusion"
    content: str
    doc_id: str
    confidence: float

@dataclass
class ReasoningChain:
    chain_id: str
    steps: List[ReasoningStep]
    chain_strength: float

@dataclass
class ContextBundle:
    core_documents: List[Dict[str, Any]]
    supporting_context: Dict[str, List[Dict[str, Any]]]
    reasoning_chains: List[ReasoningChain]
    provenance_graph: Dict[str, List[str]]
    total_tokens: int

def embed_text_dense(text: str, model: SentenceTransformer) -> List[float]:
    return model.encode(text).tolist()

def embed_text_sparse(text: str, tokenizer, model) -> Dict[str, List]:
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs).logits.squeeze(0)
    scores = torch.log(1 + torch.relu(outputs))
    max_scores, _ = torch.max(scores, dim=0)
    non_zero_indices = torch.nonzero(max_scores).squeeze(1).tolist()
    non_zero_values = max_scores[non_zero_indices].tolist()
    tokens = tokenizer.convert_ids_to_tokens(non_zero_indices)
    indices = tokenizer.convert_tokens_to_ids(tokens)
    return {
        "values": non_zero_values,
        "indices": indices,
    }

class QueryAnalyzer:
    def __init__(self):
        self.legal_domain_keywords = {
            "contract": ["contract", "agreement", "breach", "consideration", "offer", "acceptance"],
            "tort": ["negligence", "liability", "damages", "duty", "breach"],
            "criminal": ["criminal", "offense", "punishment", "conviction", "sentence"],
            "constitutional": ["constitutional", "fundamental rights", "directive principles"],
            "property": ["property", "ownership", "title", "possession", "transfer"]
        }
        
        self.intent_patterns = {
            QueryIntent.DOCTRINAL: ["interpret", "meaning", "definition", "scope", "provisions"],
            QueryIntent.PRECEDENTIAL: ["precedent", "case law", "judicial", "ruling", "decided"],
            QueryIntent.PROCEDURAL: ["procedure", "process", "steps", "filing", "court"],
            QueryIntent.COMPARATIVE: ["compare", "difference", "similar", "contrast", "versus"]
        }

    def analyze_query(self, query: str) -> QueryContext:
        """Comprehensive query analysis with intent classification"""
        intent = self._classify_intent(query)
        complexity = self._assess_complexity(query)
        domains = self._extract_legal_domains(query)
        jurisdictions = self._extract_jurisdictions(query)
        entities = self._extract_legal_entities(query)
        
        return QueryContext(
            raw_query=query,
            intent=intent,
            complexity_score=complexity,
            legal_domains=domains,
            jurisdictions=jurisdictions,
            extracted_entities=entities
        )
    
    def _classify_intent(self, query: str) -> QueryIntent:
        """Classify query intent using pattern matching"""
        query_lower = query.lower()
        intent_scores = {}
        
        for intent, patterns in self.intent_patterns.items():
            score = sum(1 for pattern in patterns if pattern in query_lower)
            intent_scores[intent] = score
        
        if not intent_scores or max(intent_scores.values()) == 0:
            return QueryIntent.MIXED
        
        return max(intent_scores, key=intent_scores.get)
    
    def _assess_complexity(self, query: str) -> float:
        """Assess query complexity based on various factors"""
        complexity_indicators = [
            len(query.split()) > 15,  # Long queries
            "and" in query.lower() or "or" in query.lower(),  # Logical operators
            query.count("?") > 1,  # Multiple questions
            any(word in query.lower() for word in ["compare", "analyze", "explain", "distinguish"]),
            re.search(r'\d{4}', query) is not None,  # Year mentions
        ]
        return sum(complexity_indicators) / len(complexity_indicators)
    
    def _extract_legal_domains(self, query: str) -> List[str]:
        """Extract legal domains from query"""
        query_lower = query.lower()
        domains = []
        
        for domain, keywords in self.legal_domain_keywords.items():
            if any(keyword in query_lower for keyword in keywords):
                domains.append(domain)
        
        return domains
    
    def _extract_jurisdictions(self, query: str) -> List[str]:
        """Extract jurisdictions mentioned in query"""
        jurisdictions = []
        jurisdiction_patterns = [
            r'supreme court', r'high court', r'district court',
        ]
        
        for pattern in jurisdiction_patterns:
            if re.search(pattern, query.lower()):
                jurisdictions.append(pattern.replace(r'\b', '').replace(r'\s+', ' '))
        
        return jurisdictions
    
    def _extract_legal_entities(self, query: str) -> List[str]:
        """Extract legal entities like case names, act names"""
        entities = []
        
        # Pattern for case citations (simplified)
        case_pattern = r'([A-Z][a-z]+ v\.? [A-Z][a-z]+)'
        cases = re.findall(case_pattern, query)
        entities.extend(cases)
        
        # Pattern for act names
        act_pattern = r'([A-Z][A-Za-z\s]+ Act,?\s*\d{4})'
        acts = re.findall(act_pattern, query)
        entities.extend(acts)
        
        return entities

class VectorDatabase(ABC):
    @abstractmethod
    async def search(self, query: str, top_k: int = 50) -> List[RetrievedChunk]:
        pass

class DenseVectorDB(VectorDatabase):
    def __init__(self, db_type: DocumentType, pinecone_client: Pinecone, model: SentenceTransformer):
        self.db_type = db_type
        self.model = model
        # Choose the correct index based on document type
        if self.db_type == DocumentType.CASE:
            self.index = pinecone_client.Index(PINECONE_CASE_DENSE_INDEX)
        else:
            self.index = pinecone_client.Index(PINECONE_ACTS_DENSE_INDEX)
        logger.info(f"Initialized DenseVectorDB for {db_type.value} with index: {self.index.describe_index_stats()}")

    async def search(self, query: str, top_k: int = 50) -> List[RetrievedChunk]:
        """Performs a real dense vector search using Pinecone."""
        logger.info(f"Searching dense {self.db_type.value} DB for: {query}")
        try:
            query_vector = embed_text_dense(query, self.model)
            query_response = self.index.query(
                vector=query_vector,
                top_k=top_k,
                include_metadata=True
            )
            
            # --- Conversion from Pinecone result to RetrievedChunk dataclass ---
            results = []
            for match in query_response.get('matches', []):
                metadata = match.get('metadata', {})
                doc_metadata = DocumentMetadata(
                    doc_id=metadata.get('doc_id', 'unknown_doc').split('_')[0],  # Extracting doc_id from composite key
                    doc_type=self.db_type,
                    title=metadata.get('title', 'Untitled'),
                    court_level=metadata.get('court_level'),
                    jurisdiction=metadata.get('jurisdiction', ''),
                    date=datetime.fromisoformat(metadata.get('date')) if metadata.get('date') else None,
                    legal_domains=metadata.get('legal_domains', []),
                    act_sections=metadata.get('act_sections', [])
                )
                chunk = RetrievedChunk(
                    doc_id=metadata.get('doc_id', 'unknown_doc'),
                    chunk_id=match.get('id', 'unknown_chunk'),
                    content=metadata.get('content', ''),
                    metadata=doc_metadata,
                    vector_score=match.get('score', 0.0),
                    chunk_index=metadata.get('chunk_index', -1)
                )
                results.append(chunk)
            
            logger.info(f"Found {len(results)} dense results.")
            return results
        except Exception as e:
            logger.error(f"Error in DenseVectorDB search for '{query}': {e}")
            return []

class SparseVectorDB(VectorDatabase):
    def __init__(self, db_type: DocumentType, pinecone_client: Pinecone, tokenizer, model):
        self.db_type = db_type
        self.tokenizer = tokenizer
        self.model = model
        # Choose the correct index
        if self.db_type == DocumentType.CASE:
            self.index = pinecone_client.Index(PINECONE_CASE_SPARSE_INDEX)
        else:
            self.index = pinecone_client.Index(PINECONE_ACTS_SPARSE_INDEX)
        logger.info(f"Initialized SparseVectorDB for {db_type.value} with index: {self.index.describe_index_stats()}")

    async def search(self, query: str, top_k: int = 50) -> List[RetrievedChunk]:
        """Performs a real sparse vector search using Pinecone."""
        logger.info(f"Searching sparse {self.db_type.value} DB for: {query}")
        try:
            sparse_vector = embed_text_sparse(query, self.tokenizer, self.model)
            query_response = self.index.query(
                sparse_vector=sparse_vector,
                top_k=top_k,
                include_metadata=True
            )
            
            # --- Conversion from Pinecone result to RetrievedChunk dataclass ---
            results = []
            for match in query_response.get('matches', []):
                metadata = match.get('metadata', {})
                doc_metadata = DocumentMetadata(
                    doc_id=metadata.get('doc_id', 'unknown_doc').split('_')[0],  # Extracting doc_id from composite key
                    doc_type=self.db_type,
                    title=metadata.get('title', 'Untitled'),
                    court_level=metadata.get('court_level'),
                    jurisdiction=metadata.get('jurisdiction', ''),
                    date=datetime.fromisoformat(metadata.get('date')) if metadata.get('date') else None,
                    legal_domains=metadata.get('legal_domains', []),
                    act_sections=metadata.get('act_sections', [])
                )
                chunk = RetrievedChunk(
                    doc_id=metadata.get('doc_id', 'unknown_doc'),
                    content=metadata.get('content', ''),
                    metadata=doc_metadata,
                    vector_score=match.get('score', 0.0),
                    chunk_index=metadata.get('chunk_index', -1),
                    chunk_id=match.get('id', 'unknown_chunk')
                )
                results.append(chunk)

            logger.info(f"Found {len(results)} sparse results.")
            return results
        except Exception as e:
            logger.error(f"Error in SparseVectorDB search for '{query}': {e}")
            return []

class KnowledgeGraph:
    def __init__(self, driver: GraphDatabase.driver):
        self.driver = driver
        self.driver.verify_connectivity()
        logger.info("Neo4j connection verified.")

    def _execute_query(self, query, params=None):
        """Helper function to run a read query."""
        with self.driver.session(database=NEO4J_DATABASE) as session:
            result = session.run(query, parameters=params or {})
            return [dict(record) for record in result]

    def compute_kg_features(self, doc_id: str) -> KGFeatures:
        """Compute comprehensive KG features for a document from Neo4j."""
        features = KGFeatures()
        
        # This query fetches pre-computed scores and citation counts
        cypher_query = """
        MATCH (d {doc_id: $doc_id})
        OPTIONAL MATCH (d)-[:REFERS_TO]->(acts)
        WITH d,
            count(acts) AS citation_count,
            count(CASE WHEN 'Act' IN labels(acts) THEN acts END) AS act_references
        RETURN
            COALESCE(d.pagerank, 0.0) AS pagerank_score,
            COALESCE(d.betweenness, 0.0) AS betweenness_centrality,
            COALESCE(d.authority, 0.0) AS authority_score,
            citation_count,
            act_references
        """
        params = {"doc_id": doc_id}
        
        try:
            result = self._execute_query(cypher_query, params)
            if not result:
                return features
            
            data = result[0]
            features.pagerank_score = data.get('pagerank_score', 0.0)
            features.betweenness_centrality = data.get('betweenness_centrality', 0.0)
            features.authority_score = data.get('authority_score', 0.0)
            features.citation_count = data.get('citation_count', 0)
            
            # You can add more complex feature calculations here if needed
            court_level = data.get('court_level', 3)
            features.jurisdictional_weight = 1.0 / court_level if court_level else 0.33

        except Exception as e:
            logger.error(f"Error computing KG features for {doc_id}: {e}")
        
        return features
    
    def _compute_recent_citation_boost(self, doc_id: str) -> float:
        """Compute boost based on recent citations"""
        # Placeholder - in reality, check citation dates
        recent_citations = len([n for n in self.graph.predecessors(doc_id)]) * 0.1
        return min(recent_citations, 1.0)
    
    def _count_cross_domain_citations(self, doc_id: str) -> int:
        """Count citations across different legal domains"""
        # Placeholder implementation
        return np.random.randint(0, 5)
    
    def _compute_authority_score(self, features: KGFeatures) -> float:
        """Compute overall authority score"""
        return (
            0.3 * features.pagerank_score + 
            0.2 * features.betweenness_centrality +
            0.2 * min(features.citation_count / 10.0, 1.0) +
            0.15 * features.recent_citation_boost +
            0.15 * features.jurisdictional_weight
        )
    
    def get_metadata(self, doc_id: str) -> Optional[DocumentMetadata]:
        """Get document metadata from graph"""
        if doc_id not in self.graph:
            return None
            
        node_data = self.graph.nodes[doc_id]
        doc_type = DocumentType.CASE if node_data.get('type') == 'case' else DocumentType.ACT
        
        return DocumentMetadata(
            doc_id=doc_id,
            doc_type=doc_type,
            title=f"Document {doc_id}",
            court_level=np.random.randint(1, 4) if doc_type == DocumentType.CASE else None
        )
    
    def expand_context(self, doc_ids: List[str], max_depth: int = 2) -> Dict[str, List[Dict[str, Any]]]:
        """Intelligent context expansion via Neo4j graph traversal."""
        expanded_context = {
            "cited_authorities": [], # Docs that the core docs cite
            "interpretive_cases": []  # Docs that cite the core docs
        }

        # Query for documents cited BY the core set
        cited_query = """
        UNWIND $doc_ids AS core_id
        MATCH (core {doc_id: core_id})-[r]->(cited)
        RETURN cited.doc_id AS doc_id, type(r) as relation, r.strength as strength
        LIMIT 20
        """
        
        # Query for documents that CITE the core set
        interpretive_query = """
        UNWIND $doc_ids AS core_id
        MATCH (interpretive)-[r]->(core {doc_id: core_id})
        RETURN interpretive.doc_id AS doc_id, type(r) as relation, r.strength as strength
        LIMIT 20
        """
        
        try:
            cited_results = self._execute_query(cited_query, {"doc_ids": doc_ids})
            interpretive_results = self._execute_query(interpretive_query, {"doc_ids": doc_ids})

            expanded_context["cited_authorities"] = cited_results
            expanded_context["interpretive_cases"] = interpretive_results

        except Exception as e:
            logger.error(f"Error expanding context in KG: {e}")
            
        return expanded_context
    
    def extract_reasoning_chains(self, doc_ids: List[str]) -> List[ReasoningChain]:
        """Extract logical reasoning chains from graph paths"""
        chains = []
        
        for i, doc_id in enumerate(doc_ids[:3]):  # Limit to top 3 for reasoning
            if doc_id not in self.graph:
                continue
                
            # Simple chain: current doc -> cited authority -> application
            chain_steps = []
            
            # Major premise (if it's an act or citing an act)
            cited_acts = [n for n in self.graph.neighbors(doc_id) 
                         if self.graph.nodes[n].get('type') == 'act']
            
            if cited_acts:
                chain_steps.append(ReasoningStep(
                    step_type="major_premise",
                    content=f"Legal provision from {cited_acts[0]}",
                    doc_id=cited_acts[0],
                    confidence=0.9
                ))
            
            # Application (current document)
            chain_steps.append(ReasoningStep(
                step_type="application",
                content=f"Application in {doc_id}",
                doc_id=doc_id,
                confidence=0.8
            ))
            
            if len(chain_steps) >= 2:
                chains.append(ReasoningChain(
                    chain_id=f"chain_{i}",
                    steps=chain_steps,
                    chain_strength=sum(step.confidence for step in chain_steps) / len(chain_steps)
                ))
        
        return chains

class FirestoreClient:
    """
    A client to interact with Google Cloud Firestore for fetching
    pre-computed summaries and argument structures.
    """
    def __init__(self, project_id: Optional[str] = None, credential_path: Optional[str] = None):
        """
        Initializes the Firestore client.
        
        Args:
            project_id (str, optional): Your Google Cloud project ID.
            credential_path (str, optional): Path to your service account JSON key file.
        """
        self.db: BaseClient
        try:
            # Check if the app is already initialized to prevent errors on re-runs
            if not firebase_admin._apps:
                if credential_path:
                    cred = credentials.Certificate(credential_path)
                    firebase_admin.initialize_app(cred, {'projectId': project_id})
                else:
                    # If no path is provided, it uses Application Default Credentials
                    firebase_admin.initialize_app()
            
            self.db = firestore.client()
            self.presummaries_ref = self.db.collection('presummaries')
            self.claim_premise_ref = self.db.collection('claim_premise_data')
            
            logger.info("Firestore client initialized successfully.")
        except Exception as e:
            logger.error(f"Failed to initialize Firestore client: {e}")
            raise

    async def get_presummary(self, doc_id: str) -> Optional[Dict[str, Any]]:
        """
        Asynchronously get a pre-computed summary for a document from Firestore.
        
        Args:
            doc_id (str): The unique identifier for the document.
            
        Returns:
            Optional[Dict[str, Any]]: The presummary data or None if not found.
        """
        try:
            doc_ref = self.presummaries_ref.document(doc_id)
            doc_snapshot = await doc_ref.get() # Use the async get method
            
            if doc_snapshot.exists:
                return doc_snapshot.to_dict()
            else:
                logger.warning(f"No presummary found in Firestore for doc_id: {doc_id}")
                return None
        except Exception as e:
            logger.error(f"Error fetching presummary for {doc_id} from Firestore: {e}")
            return None

    async def get_claim_premise_data(self, doc_id: str) -> Optional[Dict[str, Any]]:
        """
        Asynchronously get claim-premise data for a document from Firestore.
        
        Args:
            doc_id (str): The unique identifier for the document.
            
        Returns:
            Optional[Dict[str, Any]]: The claim-premise data or None if not found.
        """
        try:
            doc_ref = self.claim_premise_ref.document(doc_id)
            doc_snapshot = await doc_ref.get()
            
            if doc_snapshot.exists:
                return doc_snapshot.to_dict()
            else:
                logger.warning(f"No claim-premise data found in Firestore for doc_id: {doc_id}")
                return None
        except Exception as e:
            logger.error(f"Error fetching claim-premise data for {doc_id} from Firestore: {e}")
            return None

    # Optional: Method for batch fetching to improve performance
    async def get_batch_data(self, doc_ids: List[str]) -> Dict[str, Dict[str, Any]]:
        """
        Fetch presummary and claim-premise data for multiple doc_ids in a batch.
        
        Args:
            doc_ids (List[str]): A list of document identifiers.
            
        Returns:
            Dict[str, Dict[str, Any]]: A dictionary mapping doc_id to its combined data.
        """
        if not doc_ids:
            return {}

        presummary_refs = [self.presummaries_ref.document(doc_id) for doc_id in doc_ids]
        claim_premise_refs = [self.claim_premise_ref.document(doc_id) for doc_id in doc_ids]
        
        try:
            # Fetch all documents in parallel
            presummary_snapshots = await self.db.getAll(presummary_refs)
            claim_premise_snapshots = await self.db.getAll(claim_premise_refs)

            batch_data = {doc_id: {} for doc_id in doc_ids}

            for snapshot in presummary_snapshots:
                if snapshot.exists:
                    batch_data[snapshot.id]['presummary'] = snapshot.to_dict()
            
            for snapshot in claim_premise_snapshots:
                if snapshot.exists:
                    batch_data[snapshot.id]['claim_premise'] = snapshot.to_dict()
            
            return batch_data
        except Exception as e:
            logger.error(f"Error in batch fetch from Firestore: {e}")
            return {doc_id: {} for doc_id in doc_ids}

class AdaptiveRetriever:
    def __init__(self, pc_client, neo4j_driver, firestore_client, dense_model, sparse_tokenizer, sparse_model):
        self.dense_case_db = DenseVectorDB(DocumentType.CASE, pc_client, dense_model)
        self.sparse_case_db = SparseVectorDB(DocumentType.CASE, pc_client, sparse_tokenizer, sparse_model)
        self.dense_act_db = DenseVectorDB(DocumentType.ACT, pc_client, dense_model)
        self.sparse_act_db = SparseVectorDB(DocumentType.ACT, pc_client, sparse_tokenizer, sparse_model)
        self.dense_model = dense_model
        self.sparse_tokenizer = sparse_tokenizer
        self.sparse_model = sparse_model
        self.vector_store = {
            'dense': self.dense_case_db,
            'sparse': self.sparse_case_db
        }
        
        self.kg = KnowledgeGraph(driver=neo4j_driver)
        self.firestore = firestore_client
        
    def _determine_retrieval_weights(self, query_context: QueryContext) -> Dict[str, float]:
        """Determine retrieval weights based on query context"""
        weights = {
            "dense_case": 0.25,
            "sparse_case": 0.25,
            "dense_act": 0.25,
            "sparse_act": 0.25
        }
        
        # Adjust based on intent
        if query_context.intent == QueryIntent.DOCTRINAL:
            weights["dense_act"] = 0.4
            weights["sparse_act"] = 0.3
            weights["dense_case"] = 0.2
            weights["sparse_case"] = 0.1
        elif query_context.intent == QueryIntent.PRECEDENTIAL:
            weights["dense_case"] = 0.4
            weights["sparse_case"] = 0.3
            weights["dense_act"] = 0.2
            weights["sparse_act"] = 0.1
        
        # Adjust based on complexity
        if query_context.complexity_score > 0.7:
            # Complex queries benefit from dense retrieval
            weights["dense_case"] *= 1.2
            weights["dense_act"] *= 1.2
            weights["sparse_case"] *= 0.8
            weights["sparse_act"] *= 0.8
        
        # Normalize weights
        total = sum(weights.values())
        return {k: v/total for k, v in weights.items()}
    
    async def retrieve_candidates(self, query_context: QueryContext, top_k: int = 100) -> List[EnhancedChunk]:
        """Adaptive multi-vector retrieval"""
        weights = self._determine_retrieval_weights(query_context)
        
        # Determine how many results to get from each DB
        k_per_db = {
            "dense_case": int(top_k * weights["dense_case"]),
            "sparse_case": int(top_k * weights["sparse_case"]),
            "dense_act": int(top_k * weights["dense_act"]),
            "sparse_act": int(top_k * weights["sparse_act"])
        }
        
        # Retrieve from all databases
        tasks = [
            self.dense_case_db.search(query_context.raw_query, k_per_db["dense_case"]),
            self.sparse_case_db.search(query_context.raw_query, k_per_db["sparse_case"]),
            self.dense_act_db.search(query_context.raw_query, k_per_db["dense_act"]),
            self.sparse_act_db.search(query_context.raw_query, k_per_db["sparse_act"])
        ]
        
        results = await asyncio.gather(*tasks)
        
        # Combine and enhance with KG features
        all_chunks = []
        for chunk_list in results:
            for chunk in chunk_list:
                enhanced_chunk = EnhancedChunk(
                    doc_id=chunk.doc_id,
                    chunk_id=chunk.chunk_id,
                    content=chunk.content,
                    metadata=chunk.metadata,
                    vector_score=chunk.vector_score,
                    chunk_index=chunk.chunk_index,
                    kg_features=self.kg.compute_kg_features(chunk.doc_id)
                )
                all_chunks.append(enhanced_chunk)
        
        return all_chunks
    
    async def retrieve_documents(self, query: str, top_k: int = 5):
        """Retrieve relevant documents using both dense and sparse vectors"""
        try:
            # Get dense results
            dense_results = await self.vector_store['dense'].search(
                query=query,
                top_k=top_k,
            )

            # Get sparse results
            sparse_results = await self.vector_store['sparse'].search(
                query=query,
                top_k=top_k,
            )

            # Combine and deduplicate results
            all_results = []
            seen_ids = set()

            # Dense and sparse results are already lists of results
            for result in dense_results + sparse_results:
                doc_id = result.doc_id  # Access as attribute
                if doc_id not in seen_ids:
                    seen_ids.add(doc_id)
                    all_results.append({
                        'id': doc_id,
                        'score': result.vector_score,
                        'metadata': result.metadata if hasattr(result, 'metadata') else {}
                    })

            # Sort by score
            all_results.sort(key=lambda x: x['score'], reverse=True)
            return all_results[:top_k]

        except Exception as e:
            logger.error(f"Error in retrieve_documents: {e}")
            raise

    async def get_graph_context(self, doc_ids: List[str]):
        """Get related information from graph database"""
        try:
            graph_context = []
            with self.kg.driver.session(database=NEO4J_DATABASE) as session:
                for doc_id in doc_ids:
                    # Query for related Acts
                    cypher_query = """
                    MATCH (n:Case)
                    WHERE n.doc_id = $id
                    MATCH (n)-[:REFERS_TO]->(related_act:Act)
                    RETURN related_act.std_id AS act_id,
                        related_act.name AS act_name
                    """
                    results = session.run(cypher_query, {"id": doc_id})
                    for record in results:
                        graph_context.append({
                            'type': 'act_reference',
                            'source_id': doc_id,
                            'act_id': record['act_id'],
                            'act_name': record['act_name']
                        })
            return graph_context

        except Exception as e:
            logger.error(f"Error in get_graph_context: {e}")
            raise

class EnhancedReranker:
    def __init__(self):
        self.feature_weights = {
            "vector_score": 0.4,
            "authority_score": 0.3,
            "metadata_alignment": 0.2,
            "kg_centrality": 0.1
        }
    
    def rerank(self, chunks: List[EnhancedChunk], query_context: QueryContext, top_k: int = 20) -> List[EnhancedChunk]:
        """Enhanced reranking with multiple signals"""
        
        for chunk in chunks:
            # Compute metadata alignment score
            metadata_score = self._compute_metadata_alignment(chunk, query_context)
            
            # Compute final score
            chunk.final_score = (
                self.feature_weights["vector_score"] * chunk.vector_score +
                self.feature_weights["authority_score"] * chunk.kg_features.authority_score +
                self.feature_weights["metadata_alignment"] * metadata_score +
                self.feature_weights["kg_centrality"] * chunk.kg_features.pagerank_score
            )
        
        # Sort by final score and return top-k
        chunks.sort(key=lambda x: x.final_score, reverse=True)
        return chunks[:top_k]
    
    def _compute_metadata_alignment(self, chunk: EnhancedChunk, query_context: QueryContext) -> float:
        """Compute alignment between chunk metadata and query context"""
        score = 0.0
        
        # Domain alignment
        if query_context.legal_domains:
            chunk_domains = set(chunk.metadata.legal_domains)
            query_domains = set(query_context.legal_domains)
            if chunk_domains & query_domains:
                score += 0.5
        
        # Jurisdiction alignment
        if query_context.jurisdictions and chunk.metadata.jurisdiction:
            if any(jurisdiction in chunk.metadata.jurisdiction.lower() 
                   for jurisdiction in query_context.jurisdictions):
                score += 0.3
        
        # Document type preference based on intent
        if query_context.intent == QueryIntent.DOCTRINAL and chunk.metadata.doc_type == DocumentType.ACT:
            score += 0.2
        elif query_context.intent == QueryIntent.PRECEDENTIAL and chunk.metadata.doc_type == DocumentType.CASE:
            score += 0.2
        
        return min(score, 1.0)

class ContextAssembler:
    def __init__(self, kg: KnowledgeGraph, firestore: FirestoreClient):
        self.kg = kg
        self.firestore = firestore
        
    async def assemble_context(self, chunks: List[EnhancedChunk], query_context: QueryContext) -> ContextBundle:
        """Assemble comprehensive context bundle"""
        
        # Extract document IDs for KG expansion
        doc_ids = [chunk.doc_id for chunk in chunks]
        
        # Determine expansion depth based on query complexity
        expansion_depth = 1 if query_context.complexity_score < 0.5 else 2
        
        # Expand context via KG
        expanded_context = self.kg.expand_context(doc_ids, expansion_depth)
        
        # Extract reasoning chains
        reasoning_chains = self.kg.extract_reasoning_chains(doc_ids)
        
        # Fetch presummaries and claim-premise data
        core_documents = []
        for chunk in chunks:
            presummary = await self.firestore.get_presummary(chunk.doc_id)
            claim_premise = await self.firestore.get_claim_premise_data(chunk.doc_id)
            
            doc_data = {
                "doc_id": chunk.doc_id,
                "chunk_id": chunk.chunk_id,
                "relevance_score": chunk.final_score,
                "authority_score": chunk.kg_features.authority_score,
                "presummary": presummary.get("summary", "") if presummary else "",
                "key_claims": claim_premise.get("claims", []) if claim_premise else [],
                "legal_premises": claim_premise.get("premises", []) if claim_premise else [],
                "provenance": "direct_match",
                "metadata": {
                    "doc_type": chunk.metadata.doc_type.value,
                    "title": chunk.metadata.title,
                    "court_level": chunk.metadata.court_level
                }
            }
            core_documents.append(doc_data)
        
        # Enhance supporting context with presummaries
        enhanced_supporting = {}
        for context_type, context_docs in expanded_context.items():
            enhanced_docs = []
            for doc_info in context_docs[:5]:  # Limit supporting docs
                presummary = await self.firestore.get_presummary(doc_info["doc_id"])
                enhanced_doc = {
                    **doc_info,
                    "presummary": presummary.get("summary", "") if presummary else "",
                    "key_points": presummary.get("key_points", []) if presummary else []
                }
                enhanced_docs.append(enhanced_doc)
            enhanced_supporting[context_type] = enhanced_docs
        
        # Build provenance graph
        provenance_graph = {}
        for chunk in chunks:
            provenance_graph[chunk.doc_id] = [
                neighbor["doc_id"] for neighbor in 
                expanded_context.get("cited_authorities", [])[:3]
            ]
        
        # Estimate token count (rough approximation)
        total_tokens = self._estimate_tokens(core_documents, enhanced_supporting, reasoning_chains)
        
        return ContextBundle(
            core_documents=core_documents,
            supporting_context=enhanced_supporting,
            reasoning_chains=reasoning_chains,
            provenance_graph=provenance_graph,
            total_tokens=total_tokens
        )
    
    def _estimate_tokens(self, core_docs: List[Dict], supporting: Dict, chains: List[ReasoningChain]) -> int:
        """Rough token estimation"""
        core_tokens = sum(len(doc.get("presummary", "").split()) * 1.3 for doc in core_docs)
        supporting_tokens = sum(
            sum(len(doc.get("presummary", "").split()) * 1.3 for doc in docs)
            for docs in supporting.values()
        )
        chain_tokens = sum(
            sum(len(step.content.split()) * 1.3 for step in chain.steps)
            for chain in chains
        )
        
        return int(core_tokens + supporting_tokens + chain_tokens)

class EnhancedLegalRAGSystem:
    def __init__(self, 
                 dense_model=None,
                 sparse_model=None,
                 sparse_tokenizer=None,
                 pinecone_client=None,
                 neo4j_driver=None,
                 firestore_project_id=None,
                 firestore_credentials=None):
        """Initialize the enhanced RAG system with models and database connections"""
        self.dense_model = dense_model
        self.sparse_model = sparse_model
        self.sparse_tokenizer = sparse_tokenizer
        self.pinecone_client = pinecone_client
        self.neo4j_driver = neo4j_driver
        self.firestore_project_id = firestore_project_id
        self.firestore_credentials = firestore_credentials
        self.reranker = EnhancedReranker()
        self.context_assembler = ContextAssembler(
            kg=KnowledgeGraph(driver=self.neo4j_driver),
            firestore=FirestoreClient(
                project_id=self.firestore_project_id,
                credential_path=self.firestore_credentials
            )
        )
        
        # Initialize sub-components
        self.query_analyzer = QueryAnalyzer()
        self.vector_store = self._init_vector_store()
        self.graph_store = self._init_graph_store()
        # self.argument_analyzer = ArgumentAnalyzer()

        # Add this after initializing vector_store and graph_store:
        self.retriever = AdaptiveRetriever(
            # vector_store=self.vector_store,
            # graph_store=self.graph_store,
            dense_model=self.dense_model,
            sparse_model=self.sparse_model,
            sparse_tokenizer=self.sparse_tokenizer,
            pc_client=self.pinecone_client,
            neo4j_driver=self.neo4j_driver,
            firestore_client=FirestoreClient(
                project_id=self.firestore_project_id,
                credential_path=self.firestore_credentials
            )
        )

    def _init_vector_store(self):
        """Initialize vector store connections"""
        if not self.pinecone_client:
            raise ValueError("Pinecone client is required")
        return {
            'dense': self.pinecone_client.Index(PINECONE_CASE_DENSE_INDEX),
            'sparse': self.pinecone_client.Index(PINECONE_CASE_SPARSE_INDEX)
        }

    def _init_graph_store(self):
        """Initialize graph database connection"""
        if not self.neo4j_driver:
            raise ValueError("Neo4j driver is required")
        return self.neo4j_driver
    
    async def process_query(self, query: str, max_tokens: int = 99999) -> ContextBundle:
        """Main processing pipeline"""
        logger.info(f"Processing query: {query}")
        
        # Step 1: Analyze query
        query_context = self.query_analyzer.analyze_query(query)
        logger.info(f"Query intent: {query_context.intent}, complexity: {query_context.complexity_score:.2f}")
                
        # Step 2: Adaptive retrieval
        candidates = await self.retriever.retrieve_candidates(query_context, top_k=100)
        logger.info(f"Retrieved {len(candidates)} candidates")
        
        # Step 3: Enhanced reranking
        top_chunks = self.reranker.rerank(candidates, query_context, top_k=20)
        logger.info(f"Reranked to top {len(top_chunks)} chunks")
        
        # Step 4: Context assembly
        context_bundle = await self.context_assembler.assemble_context(top_chunks, query_context)
        logger.info(f"Assembled context with {context_bundle.total_tokens} estimated tokens")
        
        # Step 5: Token budget management
        if context_bundle.total_tokens > max_tokens:
            context_bundle = self._trim_context(context_bundle, max_tokens)
        
        return context_bundle
    
    def _trim_context(self, context_bundle: ContextBundle, max_tokens: int) -> ContextBundle:
        """Intelligently trim context to fit token budget"""
        logger.info(f"Trimming context from {context_bundle.total_tokens} to {max_tokens} tokens")
        
        # Priority order: core documents > reasoning chains > supporting context
        target_core_ratio = 0.6
        target_chain_ratio = 0.25
        target_support_ratio = 0.15
        
        core_budget = int(max_tokens * target_core_ratio)
        chain_budget = int(max_tokens * target_chain_ratio)
        support_budget = int(max_tokens * target_support_ratio)
        
        # Trim core documents (keep highest scoring)
        trimmed_core = []
        current_tokens = 0
        for doc in sorted(context_bundle.core_documents, key=lambda x: x["relevance_score"], reverse=True):
            doc_tokens = len(doc.get("presummary", "").split()) * 1.3
            if current_tokens + doc_tokens <= core_budget:
                trimmed_core.append(doc)
                current_tokens += doc_tokens
            else:
                break
        
        # Trim reasoning chains (keep strongest)
        trimmed_chains = []
        current_tokens = 0
        for chain in sorted(context_bundle.reasoning_chains, key=lambda x: x.chain_strength, reverse=True):
            chain_tokens = sum(len(step.content.split()) * 1.3 for step in chain.steps)
            if current_tokens + chain_tokens <= chain_budget:
                trimmed_chains.append(chain)
                current_tokens += chain_tokens
            else:
                break
        
        # Trim supporting context
        trimmed_supporting = {}
        for context_type, docs in context_bundle.supporting_context.items():
            trimmed_supporting[context_type] = docs[:2]  # Keep top 2 per category
        
        return ContextBundle(
            core_documents=trimmed_core,
            supporting_context=trimmed_supporting,
            reasoning_chains=trimmed_chains,
            provenance_graph=context_bundle.provenance_graph,
            total_tokens=self.context_assembler._estimate_tokens(
                trimmed_core, trimmed_supporting, trimmed_chains
            )
        )

# Alternative Approaches and Extensions

class GraphFirstRetriever(AdaptiveRetriever):
    """Alternative approach: Start with graph entities, then retrieve"""
    
    async def retrieve_candidates(self, query_context: QueryContext, top_k: int = 100) -> List[EnhancedChunk]:
        """Graph-first retrieval approach"""
        
        # Step 1: Extract entities from query
        query_entities = self._extract_entities_from_query(query_context.raw_query)
        
        # Step 2: Find related entities in KG
        expanded_entities = self._expand_entities_via_kg(query_entities)
        
        # Step 3: Targeted vector search on entity-related documents
        targeted_chunks = await self._targeted_vector_search(
            query_context, expanded_entities, top_k
        )
        
        return targeted_chunks
    
    def _extract_entities_from_query(self, query: str) -> List[str]:
        """Extract legal entities that might exist in KG"""
        # Use NER or pattern matching to find potential entity mentions
        entities = []
        
        # Look for case name patterns
        case_patterns = [
            r'([A-Z][a-z]+ v\.? [A-Z][a-z]+)', 
            r'([A-Z][a-z]+ vs\.? [A-Z][a-z]+)'
        ]
        for pattern in case_patterns:
            matches = re.findall(pattern, query)
            entities.extend(matches)
        
        # Look for act patterns
        act_patterns = [
            r'([A-Z][A-Za-z\s]+ Act,?\s*\d{4})',
            r'(Section \d+)',
            r'(Article \d+)'
        ]
        for pattern in act_patterns:
            matches = re.findall(pattern, query)
            entities.extend(matches)
        
        return entities
    
    def _expand_entities_via_kg(self, entities: List[str]) -> List[str]:
        """Expand entities using knowledge graph relationships"""
        expanded = set(entities)
        
        for entity in entities:
            # Find similar entities in KG (simplified matching)
            for node in self.kg.graph.nodes():
                if any(term.lower() in node.lower() for term in entity.split()):
                    expanded.add(node)
                    # Add one-hop neighbors
                    neighbors = list(self.kg.graph.neighbors(node))
                    expanded.update(neighbors[:3])  # Limit expansion
        
        return list(expanded)
    
    async def _targeted_vector_search(self, query_context: QueryContext, 
                                    entities: List[str], top_k: int) -> List[EnhancedChunk]:
        """Perform vector search focused on entity-related documents"""
        # Create entity-enhanced query
        enhanced_query = f"{query_context.raw_query} {' '.join(entities[:5])}"
        
        # Use regular retrieval with enhanced query
        return await super().retrieve_candidates(
            QueryContext(
                raw_query=enhanced_query,
                intent=query_context.intent,
                complexity_score=query_context.complexity_score,
                legal_domains=query_context.legal_domains,
                jurisdictions=query_context.jurisdictions,
                extracted_entities=entities
            ),
            top_k
        )

class HierarchicalRetriever(AdaptiveRetriever):
    """Alternative approach: Hierarchical retrieval with multiple stages"""
    
    async def retrieve_candidates(self, query_context: QueryContext, top_k: int = 100) -> List[EnhancedChunk]:
        """Multi-stage hierarchical retrieval"""
        
        # Stage 1: Broad retrieval
        stage1_candidates = await self._stage1_broad_retrieval(query_context, top_k * 3)
        
        # Stage 2: Focused re-retrieval based on stage 1 results
        stage2_candidates = await self._stage2_focused_retrieval(
            query_context, stage1_candidates, top_k * 2
        )
        
        # Stage 3: KG-enhanced refinement
        final_candidates = await self._stage3_kg_refinement(
            query_context, stage2_candidates, top_k
        )
        
        return final_candidates
    
    async def _stage1_broad_retrieval(self, query_context: QueryContext, top_k: int) -> List[EnhancedChunk]:
        """Stage 1: Cast wide net with basic retrieval"""
        # Use equal weights for broad coverage
        return await super().retrieve_candidates(query_context, top_k)
    
    async def _stage2_focused_retrieval(self, query_context: QueryContext, 
                                      stage1_results: List[EnhancedChunk], 
                                      top_k: int) -> List[EnhancedChunk]:
        """Stage 2: Focus on promising areas identified in stage 1"""
        
        # Analyze stage 1 results to identify key themes
        top_domains = self._extract_dominant_domains(stage1_results)
        top_authorities = self._extract_top_authorities(stage1_results)
        
        # Create focused query
        domain_terms = " ".join(top_domains[:3])
        authority_terms = " ".join(top_authorities[:2])
        focused_query = f"{query_context.raw_query} {domain_terms} {authority_terms}"
        
        # Targeted retrieval with adjusted weights
        adjusted_context = QueryContext(
            raw_query=focused_query,
            intent=query_context.intent,
            complexity_score=query_context.complexity_score,
            legal_domains=top_domains,
            jurisdictions=query_context.jurisdictions
        )
        
        return await super().retrieve_candidates(adjusted_context, top_k)
    
    async def _stage3_kg_refinement(self, query_context: QueryContext,
                                  stage2_results: List[EnhancedChunk],
                                  top_k: int) -> List[EnhancedChunk]:
        """Stage 3: Use KG to refine and expand promising candidates"""
        
        # Get KG expansion for top candidates
        top_docs = [chunk.doc_id for chunk in stage2_results[:top_k//2]]
        kg_expansion = self.kg.expand_context(top_docs, max_depth=1)
        
        # Include KG-suggested documents
        kg_suggested_docs = []
        for context_type, docs in kg_expansion.items():
            kg_suggested_docs.extend([doc["doc_id"] for doc in docs[:3]])
        
        # Combine stage 2 results with KG suggestions
        combined_results = stage2_results[:top_k//2]  # Keep top half
        
        # Add KG suggestions as new candidates
        for doc_id in kg_suggested_docs:
            if not any(chunk.doc_id == doc_id for chunk in combined_results):
                # Create pseudo-chunk for KG suggestion
                metadata = self.kg.get_metadata(doc_id)
                if metadata:
                    kg_chunk = EnhancedChunk(
                        doc_id=doc_id,
                        chunk_id=f"kg_suggested_{doc_id}",
                        content=f"KG suggested document: {doc_id}",
                        metadata=metadata,
                        vector_score=0.7,  # Default score for KG suggestions
                        chunk_index=0,
                        kg_features=self.kg.compute_kg_features(doc_id)
                    )
                    combined_results.append(kg_chunk)
        
        return combined_results[:top_k]
    
    def _extract_dominant_domains(self, chunks: List[EnhancedChunk]) -> List[str]:
        """Extract dominant legal domains from chunk results"""
        domain_counts = {}
        for chunk in chunks[:10]:  # Analyze top 10
            for domain in chunk.metadata.legal_domains:
                domain_counts[domain] = domain_counts.get(domain, 0) + 1
        
        return sorted(domain_counts.keys(), key=domain_counts.get, reverse=True)
    
    def _extract_top_authorities(self, chunks: List[EnhancedChunk]) -> List[str]:
        """Extract top authorities/courts from chunk results"""
        authorities = []
        for chunk in chunks[:5]:  # Top 5 chunks
            if chunk.metadata.court_level and chunk.metadata.court_level <= 2:
                authorities.append(chunk.metadata.title.split()[0])  # First word as authority
        
        return list(set(authorities))

# Advanced Context Assembly Strategies

class ArgumentAwareContextAssembler(ContextAssembler):
    """Enhanced context assembler focusing on legal argumentation"""
    
    async def assemble_context(self, chunks: List[EnhancedChunk], query_context: QueryContext) -> ContextBundle:
        """Assemble context with focus on argumentative structure"""
        
        # Regular assembly
        base_context = await super().assemble_context(chunks, query_context)
        
        # Enhanced reasoning chain construction
        enhanced_chains = await self._build_enhanced_reasoning_chains(chunks, query_context)
        
        # Argument structure analysis
        argument_analysis = await self._analyze_argument_structures(chunks)
        
        # Contradiction detection
        contradictions = await self._detect_contradictions(chunks)
        
        # Enhanced context bundle
        enhanced_bundle = ContextBundle(
            core_documents=base_context.core_documents,
            supporting_context={
                **base_context.supporting_context,
                "argument_analysis": argument_analysis,
                "contradictions": contradictions
            },
            reasoning_chains=enhanced_chains,
            provenance_graph=base_context.provenance_graph,
            total_tokens=base_context.total_tokens
        )
        
        return enhanced_bundle
    
    async def _build_enhanced_reasoning_chains(self, chunks: List[EnhancedChunk], 
                                             query_context: QueryContext) -> List[ReasoningChain]:
        """Build more sophisticated reasoning chains"""
        chains = []
        
        # Group chunks by legal reasoning patterns
        statutory_chunks = [c for c in chunks if c.metadata.doc_type == DocumentType.ACT]
        case_chunks = [c for c in chunks if c.metadata.doc_type == DocumentType.CASE]
        
        # Build syllogistic reasoning chains
        for i, case_chunk in enumerate(case_chunks[:3]):
            chain_steps = []
            
            # Find applicable statute
            applicable_statutes = await self._find_applicable_statutes(case_chunk, statutory_chunks)
            
            if applicable_statutes:
                statute = applicable_statutes[0]
                
                # Major premise (statute)
                claim_premise = await self.firestore.get_claim_premise_data(statute.doc_id)
                major_premise_content = (claim_premise.get("premises", [""])[0] 
                                       if claim_premise else f"Legal rule from {statute.metadata.title}")
                
                chain_steps.append(ReasoningStep(
                    step_type="major_premise",
                    content=major_premise_content,
                    doc_id=statute.doc_id,
                    confidence=0.9
                ))
                
                # Minor premise (facts/application)
                case_claim_premise = await self.firestore.get_claim_premise_data(case_chunk.doc_id)
                minor_premise_content = (case_claim_premise.get("premises", [""])[0] 
                                       if case_claim_premise else f"Facts from {case_chunk.metadata.title}")
                
                chain_steps.append(ReasoningStep(
                    step_type="minor_premise",
                    content=minor_premise_content,
                    doc_id=case_chunk.doc_id,
                    confidence=0.8
                ))
                
                # Conclusion
                conclusion_content = (case_claim_premise.get("claims", [""])[0] 
                                    if case_claim_premise else f"Holding in {case_chunk.metadata.title}")
                
                chain_steps.append(ReasoningStep(
                    step_type="conclusion",
                    content=conclusion_content,
                    doc_id=case_chunk.doc_id,
                    confidence=0.85
                ))
                
                chains.append(ReasoningChain(
                    chain_id=f"syllogistic_chain_{i}",
                    steps=chain_steps,
                    chain_strength=sum(step.confidence for step in chain_steps) / len(chain_steps)
                ))
        
        return chains
    
    async def _find_applicable_statutes(self, case_chunk: EnhancedChunk, 
                                      statutory_chunks: List[EnhancedChunk]) -> List[EnhancedChunk]:
        """Find statutes applicable to a case"""
        applicable = []
        
        # Check KG connections
        case_neighbors = list(self.kg.graph.neighbors(case_chunk.doc_id))
        
        for statute in statutory_chunks:
            if statute.doc_id in case_neighbors:
                applicable.append(statute)
        
        # Sort by relevance score
        applicable.sort(key=lambda x: x.final_score, reverse=True)
        return applicable
    
    async def _analyze_argument_structures(self, chunks: List[EnhancedChunk]) -> List[Dict[str, Any]]:
        """Analyze and categorize argument structures"""
        argument_analysis = []
        
        for chunk in chunks[:5]:  # Analyze top 5
            claim_premise = await self.firestore.get_claim_premise_data(chunk.doc_id)
            
            if claim_premise:
                analysis = {
                    "doc_id": chunk.doc_id,
                    "argument_type": self._classify_argument_type(claim_premise),
                    "strength": self._assess_argument_strength(claim_premise),
                    "claims": claim_premise.get("claims", []),
                    "premises": claim_premise.get("premises", [])
                }
                argument_analysis.append(analysis)
        
        return argument_analysis
    
    def _classify_argument_type(self, claim_premise_data: Dict[str, Any]) -> str:
        """Classify the type of legal argument"""
        claims = claim_premise_data.get("claims", [])
        premises = claim_premise_data.get("premises", [])
        
        # Simple classification based on content patterns
        all_text = " ".join(claims + premises).lower()
        
        if any(word in all_text for word in ["precedent", "case law", "established"]):
            return "precedential"
        elif any(word in all_text for word in ["statute", "provision", "section"]):
            return "statutory"
        elif any(word in all_text for word in ["policy", "purpose", "intent"]):
            return "teleological"
        else:
            return "textual"
    
    def _assess_argument_strength(self, claim_premise_data: Dict[str, Any]) -> float:
        """Assess the strength of legal argument"""
        claims = claim_premise_data.get("claims", [])
        premises = claim_premise_data.get("premises", [])
        
        # Simple heuristic: more premises = stronger argument
        base_strength = min(len(premises) / 3.0, 1.0)
        
        # Boost for clear, specific claims
        if claims and any(len(claim.split()) > 5 for claim in claims):
            base_strength += 0.2
        
        return min(base_strength, 1.0)
    
    async def _detect_contradictions(self, chunks: List[EnhancedChunk]) -> List[Dict[str, Any]]:
        """Detect potential contradictions between documents"""
        contradictions = []
        
        # Compare top chunks pairwise
        for i, chunk1 in enumerate(chunks[:3]):
            for chunk2 in chunks[i+1:4]:
                claim_premise1 = await self.firestore.get_claim_premise_data(chunk1.doc_id)
                claim_premise2 = await self.firestore.get_claim_premise_data(chunk2.doc_id)
                
                if claim_premise1 and claim_premise2:
                    contradiction_score = self._compute_contradiction_score(
                        claim_premise1, claim_premise2
                    )
                    
                    if contradiction_score > 0.7:
                        contradictions.append({
                            "doc1": chunk1.doc_id,
                            "doc2": chunk2.doc_id,
                            "contradiction_score": contradiction_score,
                            "type": "claims_conflict"
                        })
        
        return contradictions
    
    def _compute_contradiction_score(self, cp1: Dict[str, Any], cp2: Dict[str, Any]) -> float:
        """Compute contradiction score between two claim-premise structures"""
        claims1 = set(cp1.get("claims", []))
        claims2 = set(cp2.get("claims", []))
        
        # Simple contradiction detection: look for negating words
        contradiction_indicators = ["not", "cannot", "prohibition", "invalid", "unlawful"]
        
        score = 0.0
        for claim1 in claims1:
            for claim2 in claims2:
                # Check if claims are about similar topics but with contradictory conclusions
                if any(indicator in claim1.lower() or indicator in claim2.lower() 
                       for indicator in contradiction_indicators):
                    # Simple similarity check (in practice, use embeddings)
                    common_words = set(claim1.lower().split()) & set(claim2.lower().split())
                    if len(common_words) > 2:
                        score += 0.3
        
        return min(score, 1.0)

# Usage Example and Testing
async def main():
    """Main function demonstrating the enhanced system"""
    
    # --- Initialize all clients ONCE at the start ---
    pc_client = Pinecone(api_key=PINECONE_API_KEY)
    neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))    
    dense_model_instance = SentenceTransformer(PINECONE_DENSE_EMBED_MODEL)
    sparse_tokenizer_instance = AutoTokenizer.from_pretrained(PINECONE_SPARSE_EMBED_MODEL)
    sparse_model_instance = AutoModelForMaskedLM.from_pretrained(PINECONE_SPARSE_EMBED_MODEL)

    # Initialize the system with all real clients
    rag_system = EnhancedLegalRAGSystem(
        dense_model=dense_model_instance,
        sparse_model=sparse_model_instance,
        sparse_tokenizer=sparse_tokenizer_instance,
        pinecone_client=pc_client,
        neo4j_driver=neo4j_driver,
        firestore_project_id=FIRESTORE_PROJECT_ID,
        firestore_credentials=FIRESTORE_CREDENTIALS_PATH
    )
    
    # Example legal queries
    test_queries = [
        "Who is the accused in the contempt of court case regarding bribery allegations against judges?",
        # "What article did Desmond Chathuranga De Alwis upload on lankanewsweb.org?",
        # "What charges were brought against Desmond Chathuranga De Alwis for publishing an article?",
        # "What is the content of the article published by Desmond Chathuranga De Alwis on 30.04.2020?",
        # "What is the legal basis for the contempt of court case against Desmond Chathuranga De Alwis?",
    ]
    
    # Process each query
    for i, query in enumerate(test_queries):
        print(f"\n{'='*100}")
        print(f"Processing Query {i+1}: {query}")
        print('='*100)
        
        try:
            # Process the query
            context_bundle = await rag_system.process_query(query, max_tokens=99999)
            
            # Display results
            print(f"\nQuery Analysis:")
            query_context = rag_system.query_analyzer.analyze_query(query)
            print(f"  Intent: {query_context.intent.value}")
            print(f"  Complexity: {query_context.complexity_score:.2f}")
            print(f"  Domains: {query_context.legal_domains}")
            
            print(f"\nContext Bundle Summary:")
            print(f"  Core Documents: {len(context_bundle.core_documents)}")
            print(f"  Supporting Context Types: {len(context_bundle.supporting_context)}")
            print(f"  Reasoning Chains: {len(context_bundle.reasoning_chains)}")
            print(f"  Total Tokens: {context_bundle.total_tokens}")
            
            print(f"\nTop 3 Core Documents:")
            for j, doc in enumerate(context_bundle.core_documents[:3]):
                print(f"  {j+1}. {doc['metadata']['title']} (Score: {doc['relevance_score']:.3f})")
            
            print(f"\nReasoning Chains:")
            for chain in context_bundle.reasoning_chains:
                print(f"  Chain {chain.chain_id} (Strength: {chain.chain_strength:.3f}):")
                for step in chain.steps:
                    print(f"    - {step.step_type}: {step.content[:100]}...")
            
        except Exception as e:
            print(f"Error processing query: {e}")
            logger.error(f"Error processing query '{query}': {e}", exc_info=True)

        finally:
            # Ensure the Neo4j driver is closed gracefully on exit
            if 'neo4j_driver' in locals() and neo4j_driver:
                neo4j_driver.close()
                logger.info("Neo4j driver closed.")

if __name__ == "__main__":
    nest_asyncio.apply()
    await main()

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:__main__:Neo4j connection verified.
INFO:__main__:Firestore client initialized successfully.
INFO:__main__:Firestore client initialized successfully.
INFO:__main__:Neo4j connection verified.
INFO:__main__:Firestore client initialized successfully.
INFO:__main__:Firestore client initialized successfully.
INFO:__main__:Initialized DenseVectorDB for case with index: {'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 31061}},
 'total_vector_count': 31061,
 'vector_type': 'dense'}
INFO:__main__:Initialized DenseVectorDB for case with index: {'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces':


Processing Query 1: Who is the accused in the contempt of court case regarding bribery allegations against judges?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:__main__:Found 25 dense results.
INFO:__main__:Searching sparse case DB for: Who is the accused in the contempt of court case regarding bribery allegations against judges?
INFO:__main__:Searching sparse case DB for: Who is the accused in the contempt of court case regarding bribery allegations against judges?
INFO:__main__:Found 25 sparse results.
INFO:__main__:Searching dense act DB for: Who is the accused in the contempt of court case regarding bribery allegations against judges?
INFO:__main__:Found 25 sparse results.
INFO:__main__:Searching dense act DB for: Who is the accused in the contempt of court case regarding bribery allegations against judges?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:__main__:Found 25 dense results.
INFO:__main__:Searching sparse act DB for: Who is the accused in the contempt of court case regarding bribery allegations against judges?
INFO:__main__:Searching sparse act DB for: Who is the accused in the contempt of court case regarding bribery allegations against judges?
INFO:__main__:Found 25 sparse results.
INFO:__main__:Found 25 sparse results.
INFO:__main__:Retrieved 100 candidates
INFO:__main__:Reranked to top 20 chunks
INFO:__main__:Retrieved 100 candidates
INFO:__main__:Reranked to top 20 chunks
ERROR:__main__:Error processing query 'Who is the accused in the contempt of court case regarding bribery allegations against judges?': 'KnowledgeGraph' object has no attribute 'graph'
Traceback (most recent call last):
  File "C:\Users\cheth\AppData\Local\Temp\ipykernel_13612\324448666.py", line 1477, in main
    context_bundle = await rag_system.process_query(query, max_tokens=99999)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Error processing query: 'KnowledgeGraph' object has no attribute 'graph'
